In [58]:
import pandas as pd
import numpy as np

In [6]:
# Récupération du dataset
path = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
df = pd.read_csv(path, nrows=10000, sep='\t',  encoding="utf-8", low_memory = False, na_filter = True )     
df                                

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,glycemic-index_100g,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g
0,54,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1582569031,2020-02-24T18:30:31Z,1733085204,2024-12-01T20:33:24Z,NaN,1740205422,2025-02-22T06:23:42Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,63,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1673620307,2023-01-13T14:31:47Z,1739902555,2025-02-18T18:15:55Z,waistline-app,1740360377,2025-02-24T01:26:17Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,114,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1580066482,2020-01-26T19:21:22Z,1737247862,2025-01-19T00:51:02Z,smoothie-app,1740075018,2025-02-20T18:10:18Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,http://world-en.openfoodfacts.org/product/0000...,inf,1634745456,2021-10-20T15:57:36Z,1740924602,2025-03-02T14:10:02Z,smoothie-app,1740924602,2025-03-02T14:10:02Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,105,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1572117743,2019-10-26T19:22:23Z,1738073570,2025-01-28T14:12:50Z,NaN,1740085032,2025-02-20T20:57:12Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,15899,http://world-en.openfoodfacts.org/product/0001...,macrofactor,1740049723,2025-02-20T11:08:43Z,1740049730,2025-02-20T11:08:50Z,macrofactor,1740049730,2025-02-20T11:08:50Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9996,1590000211,http://world-en.openfoodfacts.org/product/0001...,omnomnotes-app,1673311705,2023-01-10T00:48:25Z,1673312282,2023-01-10T00:58:02Z,omnomnotes-app,1707874424,2024-02-14T01:33:44Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9997,1590006450,http://world-en.openfoodfacts.org/product/0001...,usda-ndb-import,1489090355,2017-03-09T20:12:35Z,1728036862,2024-10-04T10:14:22Z,fix-code-bot,1728036862,2024-10-04T10:14:22Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9998,1590006978,http://world-en.openfoodfacts.org/product/0001...,smoothie-app,1727305008,2024-09-25T22:56:48Z,1728043141,2024-10-04T11:59:01Z,fix-code-bot,1738847008,2025-02-06T13:03:28Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:

def detect_variable_types(df):
    # Détection des variables numériques
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    # Détection des variables catégorielles
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Définition des catégories ordinales possibles (Peut changer par la suite) 
    predefined_ordinal_categories = {
        "nutri_score": ["e", "d", "c", "b", "a"]
    }

    ordinal_columns = []
    non_ordinal_columns = []

    for col in categorical_columns:
        unique_values = df[col].dropna().astype(str).str.lower().str.strip().unique()  # Nettoyage des valeurs
        
        for predefined_values in predefined_ordinal_categories.values():
            # Vérifier si la majorité des valeurs de la colonne appartiennent à la liste ordonnée
            matching_values = [val for val in unique_values if val in predefined_values]
            
            if len(matching_values) / len(unique_values) > 0.7:  # Seuil de 70% de correspondance
                ordinal_columns.append(col)
                break
        else:
            non_ordinal_columns.append(col)

    # Création du tableau pour affichage
    result_df = pd.DataFrame({
        "Type de variable": ["Catégorielle Ordinale"] * len(ordinal_columns) + 
                            ["Numérique"] * len(numeric_columns) + 
                            ["Catégorielle Non Ordinale"] * len(non_ordinal_columns),
        "Nom de la colonne":ordinal_columns + numeric_columns  + non_ordinal_columns
    })

    return result_df


# Exécuter la fonction et afficher le résultat sous forme de tableau
result_df = detect_variable_types(df) 
print(result_df)   


             Type de variable            Nom de la colonne
0       Catégorielle Ordinale             nutriscore_grade
1                   Numérique                 last_image_t
2   Catégorielle Non Ordinale                          url
3   Catégorielle Non Ordinale                      creator
4   Catégorielle Non Ordinale             created_datetime
..                        ...                          ...
67  Catégorielle Non Ordinale              image_small_url
68  Catégorielle Non Ordinale        image_ingredients_url
69  Catégorielle Non Ordinale  image_ingredients_small_url
70  Catégorielle Non Ordinale          image_nutrition_url
71  Catégorielle Non Ordinale    image_nutrition_small_url

[72 rows x 2 columns]


In [74]:
import pandas as pd
import numpy as np

#Appliquer le downcasting aux variables numériques pour optimiser la mémoire.
def downcast_variables(df, numeric_columns):
    for col in numeric_columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")
    return df

print("Mémoire avant downcasting :", df.memory_usage(deep=True).sum(), "bytes")
df = downcast_variables(df, result["Variables numériques"])  # Appliquer le downcasting
print("Mémoire après downcasting :", df.memory_usage(deep=True).sum(), "bytes")      

Mémoire avant downcasting : 54697515 bytes
Mémoire après downcasting : 54697515 bytes


In [86]:
def filter_categorical_columns(df, max_categories=50):
    # Sélection des colonnes catégorielles
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns
    
    # Initialisation des listes de colonnes
    kept_columns = []
    excluded_columns = []

    for col in categorical_columns:
        num_unique = df[col].nunique()  # Nombre de valeurs uniques dans la colonne
        
        if num_unique <= max_categories:
            kept_columns.append(col)
        else:
            excluded_columns.append(col)
    
    return {
        "Colonnes conservées": kept_columns,
        "Colonnes exclues": excluded_columns
    }

# Exemple d'utilisation :
result = filter_categorical_columns(df, max_categories=50)
print(result)


{'Colonnes conservées': ['abbreviated_product_name', 'packaging_text', 'first_packaging_code_geo', 'cities_tags', 'ingredients_analysis_tags', 'no_nutrition_data', 'nutriscore_grade', 'pnns_groups_1', 'pnns_groups_2', 'food_groups', 'food_groups_tags', 'food_groups_en', 'environmental_score_grade', 'owner'], 'Colonnes exclues': ['url', 'creator', 'created_datetime', 'last_modified_datetime', 'last_modified_by', 'last_updated_datetime', 'product_name', 'generic_name', 'quantity', 'packaging', 'packaging_tags', 'packaging_en', 'brands', 'brands_tags', 'categories', 'categories_tags', 'categories_en', 'origins', 'origins_tags', 'origins_en', 'manufacturing_places', 'manufacturing_places_tags', 'labels', 'labels_tags', 'labels_en', 'emb_codes', 'emb_codes_tags', 'purchase_places', 'stores', 'countries', 'countries_tags', 'countries_en', 'ingredients_text', 'ingredients_tags', 'allergens', 'traces', 'traces_tags', 'traces_en', 'serving_size', 'additives_tags', 'additives_en', 'states', 'sta

In [2]:
import pandas as pd
import numpy as np

def process_data(df, max_categories=50):
    # 1. Détection des types de variables
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    predefined_ordinal_categories = {
        "nutri_score": ["e", "d", "c", "b", "a"]
    }
    
    ordinal_columns = []
    non_ordinal_columns = []

    for col in categorical_columns:
        unique_values = df[col].dropna().astype(str).str.lower().str.strip().unique()
        
        for predefined_values in predefined_ordinal_categories.values():
            matching_values = [val for val in unique_values if val in predefined_values]
            if len(matching_values) / len(unique_values) > 0.7:
                ordinal_columns.append(col)
                break
        else:
            non_ordinal_columns.append(col)

    # 2. Optimisation de la mémoire avec le downcasting pour les variables numériques
    def downcast_variables(df, numeric_columns):
        for col in numeric_columns:
            if pd.api.types.is_integer_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], downcast="integer")
            elif pd.api.types.is_float_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], downcast="float")
        return df
    
    df = downcast_variables(df, numeric_columns)

    # 3. Filtrage des colonnes catégorielles avec un nombre de catégories supérieur à la limite
    def filter_categorical_columns(df, max_categories):
        kept_columns = []
        excluded_columns = []
        for col in categorical_columns:
            num_unique = df[col].nunique()
            if num_unique <= max_categories:
                kept_columns.append(col)
            else:
                excluded_columns.append(col)
        return kept_columns, excluded_columns
    
    kept_columns, excluded_columns = filter_categorical_columns(df, max_categories)
    
    # 4. Création du tableau pour afficher le type des variables
    result_df = pd.DataFrame({
        "Type de variable": ["Catégorielle Ordinale"] * len(ordinal_columns) + 
                            ["Numérique"] * len(numeric_columns) + 
                            ["Catégorielle Non Ordinale"] * len(non_ordinal_columns),
        "Nom de la colonne": ordinal_columns + numeric_columns + non_ordinal_columns
    })
    
    # Résultats
    print("Mémoire avant downcasting :", df.memory_usage(deep=True).sum(), "bytes")
    print("Mémoire après downcasting :", df.memory_usage(deep=True).sum(), "bytes")
    
    return {
        "Types de variables": result_df,
        "Colonnes conservées (catégorielles)": kept_columns,
        "Colonnes exclues (catégorielles)": excluded_columns
    }

# Exemple d'utilisation
path = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
df = pd.read_csv(path, nrows=10000, sep='\t', encoding="utf-8", low_memory=False, na_filter=True)
result = process_data(df)

# Afficher les résultats
print(result["Types de variables"])
print("Colonnes conservées : ", result["Colonnes conservées (catégorielles)"])
print("Colonnes exclues : ", result["Colonnes exclues (catégorielles)"])


Mémoire avant downcasting : 54159913 bytes
Mémoire après downcasting : 54159913 bytes
              Type de variable            Nom de la colonne
0        Catégorielle Ordinale             nutriscore_grade
1                    Numérique                         code
2                    Numérique                    created_t
3                    Numérique              last_modified_t
4                    Numérique               last_updated_t
..                         ...                          ...
201  Catégorielle Non Ordinale              image_small_url
202  Catégorielle Non Ordinale        image_ingredients_url
203  Catégorielle Non Ordinale  image_ingredients_small_url
204  Catégorielle Non Ordinale          image_nutrition_url
205  Catégorielle Non Ordinale    image_nutrition_small_url

[206 rows x 2 columns]
Colonnes conservées :  ['abbreviated_product_name', 'packaging_text', 'first_packaging_code_geo', 'cities_tags', 'ingredients_analysis_tags', 'no_nutrition_data', 'nutris